# Testing of the Entso-E (energy data) and Meteostat (weather) APIs

## Meteostat

In [ ]:
import meteostat as ms
import matplotlib.pyplot as plt
from datetime import date
import pandas as pd

In [ ]:
# Specify locations and time range
CITIES = {
    "Frankfurt": ms.Point(50.1155, 8.6842, 113),
    "Berlin": ms.Point(52.5200, 13.4050, 34),
    "Munich": ms.Point(48.1351, 11.5820, 519),
    "Cologne": ms.Point(50.9375, 6.9603, 37),
    "Hamburg": ms.Point(53.5511, 9.9937, 8),
}

START = date(2026, 1, 1)
END = date(2026, 2, 28)

# Fetch daily average, minimum and maximum temperature for each city
city_weather = {}
for city_name, point in CITIES.items():
    stations = ms.stations.nearby(point, limit=4)
    ts = ms.daily(stations, START, END)
    city_df = ms.interpolate(ts, point).fetch()
    city_weather[city_name] = city_df[[ms.Parameter.TEMP, ms.Parameter.TMIN, ms.Parameter.TMAX]].rename(
        columns={
            ms.Parameter.TEMP: "Temp",
            ms.Parameter.TMIN: "Min Temp",
            ms.Parameter.TMAX: "Max Temp",
        }
    )

# Build per-metric dataframes across all cities
temp_df = pd.DataFrame({city: metrics["Temp"] for city, metrics in city_weather.items()})
tmin_df = pd.DataFrame({city: metrics["Min Temp"] for city, metrics in city_weather.items()})
tmax_df = pd.DataFrame({city: metrics["Max Temp"] for city, metrics in city_weather.items()})

# Plot one axis per city + one axis for overall averages
fig, axes = plt.subplots(len(CITIES) + 1, 1, figsize=(12, 16), sharex=True)

for index, city_name in enumerate(CITIES):
    city_metrics = city_weather[city_name]
    axes[index].plot(city_metrics.index, city_metrics["Temp"], label="Avg Temp", linewidth=1.6)
    axes[index].plot(city_metrics.index, city_metrics["Min Temp"], label="Min Temp", linewidth=1.2)
    axes[index].plot(city_metrics.index, city_metrics["Max Temp"], label="Max Temp", linewidth=1.2)
    axes[index].set_title(f"Daily Temperatures in {city_name} ({START} - {END})")
    axes[index].set_ylabel("Temp (°C)")
    axes[index].grid(alpha=0.3)
    axes[index].legend(loc="upper left", ncol=3, fontsize=8)

avg_weather_df = pd.DataFrame(
    {
        "Temp": temp_df.mean(axis=1),
        "Min Temp": tmin_df.mean(axis=1),
        "Max Temp": tmax_df.mean(axis=1),
    }
)

axes[-1].plot(avg_weather_df.index, avg_weather_df["Temp"], color="black", linewidth=2, label="Avg Temp")
axes[-1].plot(avg_weather_df.index, avg_weather_df["Min Temp"], color="gray", linewidth=1.3, label="Min Temp")
axes[-1].plot(avg_weather_df.index, avg_weather_df["Max Temp"], color="tab:red", linewidth=1.3, label="Max Temp")
axes[-1].set_title(f"Daily Average Temperatures Across All Cities ({START} - {END})")
axes[-1].set_xlabel("Date")
axes[-1].set_ylabel("Temp (°C)")
axes[-1].grid(alpha=0.3)
axes[-1].legend(loc="upper left", ncol=3, fontsize=8)

plt.tight_layout()
plt.show()

combined_df = pd.concat(city_weather, axis=1)
combined_df[("Average", "Temp")] = avg_weather_df["Temp"]
combined_df[("Average", "Min Temp")] = avg_weather_df["Min Temp"]
combined_df[("Average", "Max Temp")] = avg_weather_df["Max Temp"]

combined_df.head()